# Book RAG Project

Your task is to build a **retrieval-augmented generation (RAG)** system that answers questions about **one book**, and shows the passages of the book that each answer is based on. Every group works on a different book from Project Gutenberg; the table in the README shows which book is yours.

![The project in two parts. Part 1, once per book: 1. get your book, 2. clean the text, 3. split it into chunks, 4. embed the chunks and store them in a Chroma vector store. Part 2, for each question: 5. retrieve the passages from the vector store, 6. generate the answer, and return the answer with its passages](assets/book_rag_project_workflow.png)

# The Business Problem

A public library wants to offer a **reading companion** on its website: readers ask a question about a book in their own words, for example "Why does the Tin Woodman want a heart?", and receive a short answer. Readers must be able to check every answer, so the companion also shows **the passages of the book that the answer is based on**. When the book does not contain the answer, the companion has to say so instead of making one up.

A large language model alone already knows a lot about famous books, but it does not cite passages, and it can confuse details or invent them, especially for less well-known books. Your RAG system retrieves the passages from the book first and lets the model answer **only from these passages**, so that every answer comes with the passages a reader can check.

Before the library builds the website, its staff want to **try the companion themselves**. Your last task is therefore a **command-line prototype**: a small script that librarians run in the terminal to ask questions about your book and read the answers with their passages.

# Data Description

The data ships with this repository in the folder `data/`: **eight books** from [Project Gutenberg](https://www.gutenberg.org/), one plain text file per book (UTF-8). Your group works with **one** of them.

| File | Title | Author | First published | Kind | Words |
| :-- | :-- | :-- | :-- | :-- | --: |
| `adventures_of_pinocchio_collodi.txt` | The Adventures of Pinocchio | Carlo Collodi | 1883 | children's novel, translated from Italian | 39,900 |
| `adventures_of_sherlock_holmes_doyle.txt` | The Adventures of Sherlock Holmes | Arthur Conan Doyle | 1892 | twelve detective stories | 104,500 |
| `antarctic_penguins_levick.txt` | Antarctic Penguins: A Study of Their Social Habits | G. Murray Levick | 1914 | natural history | 30,600 |
| `climate_in_relation_to_man_ward.txt` | Climate, Considered Especially in Relation to Man | Robert DeCourcy Ward | 1918 | climatology textbook (second edition) | 88,300 |
| `frankenstein_shelley.txt` | Frankenstein; or, The Modern Prometheus | Mary Wollstonecraft Shelley | 1818 | gothic novel | 75,000 |
| `the_time_machine_wells.txt` | The Time Machine | H. G. Wells | 1895 | science fiction novella | 32,500 |
| `tom_browns_school_days_hughes.txt` | Tom Brown's School Days | Thomas Hughes | 1857 | boarding school novel | 105,400 |
| `wonderful_wizard_of_oz_baum.txt` | The Wonderful Wizard of Oz | L. Frank Baum | 1900 | children's fantasy novel | 39,600 |

The word counts leave out the licence text. Every file is exactly as Project Gutenberg distributes it:

- It **starts with a header** (title, author, release date) and the line `*** START OF THE PROJECT GUTENBERG EBOOK ... ***`, and it **ends with** the line `*** END OF THE PROJECT GUTENBERG EBOOK ... ***`, followed by the Project Gutenberg licence. Only the text between these two lines belongs to the book.
- The lines are **wrapped after about 70 characters**, so a single sentence is spread over several lines. A blank line separates two paragraphs.
- Some books contain notes of the transcribers, a table of contents, and placeholders such as `[Illustration]`.

The books are in the public domain in the United States; the Project Gutenberg licence at the end of each file explains how they may be used.

## Task

1. **Get your book.** Set `BOOK` to the file name of your book and read it with `Path.read_text`. Print the first and last 1,500 characters: what comes before and after the book itself?
2. **Clean the text.** Keep only the text between the `*** START OF ...` and `*** END OF ...` lines, and join the wrapped lines of each paragraph with `re.sub`. Store the result in a [`Document`](https://reference.langchain.com/python/langchain-core/documents/base/Document) with the title and author in its `metadata`.
3. **Split into chunks.** Use [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) with a `chunk_size` and `chunk_overlap` of your choice, and `add_start_index=True`, so that every chunk stores its position in the book. How many chunks do you get?
4. **Embed and store.** Embed the chunks with [`HuggingFaceEmbeddings`](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings) (`sentence-transformers/all-MiniLM-L6-v2`) and store them in a [`Chroma`](https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma) vector store with `persist_directory="chroma_db"`.

> [!IMPORTANT]
> Pass your own `ids` to `add_documents`, for example `f"{BOOK}-{i}"`. Without them, every run of the cell adds all chunks again.

5. **Retrieve.** Create a retriever with `as_retriever(search_kwargs={"k": 4})` and `invoke` it with a few questions about your book. Print the chunks with their `start_index`: do they contain the answer?
6. **Generate.** Build an LCEL chain `prompt | llm | parser` from a [`ChatPromptTemplate`](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate) that tells the model to answer **only from the passages**, [`ChatGroq`](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) with `openai/gpt-oss-20b` and `temperature=0`, and a `StrOutputParser`. Write `ask(question)`, which returns the answer and the `start_index` of each passage. Try your questions from step 5, and one whose answer is not in the book.
7. **Build the command-line prototype.** Copy your code into `rag_cli.py` and complete the `while` loop described in its comments, so that it reads questions with `input()` until the user types `quit`, and prints each answer and its passage in colour. Run it with `uv run python rag_cli.py`.

> [!NOTE]
> The free Groq plan limits requests per minute. If you get a rate limit error, wait a minute and try again.

In [1]:
# Import the libraries you need for this project
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")

True

**1. Loading book!**

In [2]:
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# The file for my group's book
BOOK = "adventures_of_pinocchio_collodi.txt"
book_path = Path("data") / BOOK

raw_text = book_path.read_text(encoding="utf-8")

# We look at the first and last 1500 characters
print("=== FIRST 1500 CHARACTERS ===")
print(raw_text[:1500])
print("\n=== LAST 1500 CHARACTERS ===")
print(raw_text[-1500:])

=== FIRST 1500 CHARACTERS ===
The Project Gutenberg eBook of The Adventures of Pinocchio
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Adventures of Pinocchio

Author: Carlo Collodi

Translator: Carol Della Chiesa


        
Release date: January 12, 2006 [eBook #500]
                Most recently updated: September 28, 2020

Language: English

Other information and formats: www.gutenberg.org/ebooks/500

Credits: Produced by Charles Keller (for Tina); and David Widger


*** START OF THE PROJECT GUTENBERG EBOOK THE ADVENTURES OF PINOCCHIO ***




Produced by Charles Keller (for Ti

**2. Clean the text.**

In [3]:
from langchain_core.documents import Document

# Extract the text between the START and END markers
pattern = r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK .*?\*\*\*(.*?)\*\*\* END OF THE PROJECT GUTENBERG EBOOK"
match = re.search(pattern, raw_text, re.DOTALL)

if not match:
    raise ValueError("Could not find the beginning or end of the book using standard Gutenberg markers!")

book_body = match.group(1).strip()

# Merge line breaks within paragraphs (while preserving double line breaks)
paragraphs = [re.sub(r"\s*\n\s*", " ", p).strip() for p in book_body.split("\n\n")]
cleaned_text = "\n\n".join(p for p in paragraphs if p)

# Save to Document with metadata
doc = Document(
    page_content=cleaned_text,
    metadata={
        "source": BOOK,
        "title": "The Adventures of Pinocchio",
        "author": "Carlo Collodi"
    }
)

print(f"Length of cleaned text: {len(doc.page_content):,} characters")
print(doc.page_content[:500])  # Show the first 500 characters of the cleaned text

Length of cleaned text: 213,494 characters
Produced by Charles Keller (for Tina); and David Widger

Dashes; small checks; quick pass; gutchecked twice; jeebies; spellcheck

THE ADVENTURES OF PINOCCHIO

by C. Collodi

[Pseudonym of Carlo Lorenzini]

Translated from the Italian by Carol Della Chiesa

CHAPTER 1

How it happened that Mastro Cherry, carpenter, found a piece of wood that wept and laughed like a child.

Centuries ago there lived--

“A king!” my little readers will say immediately.

No, children, you are mistaken. Once upon a ti


**3. Split into chunks.**

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    add_start_index=True
)

chunks = text_splitter.split_documents([doc])
print(f"Total chunks created: {len(chunks)}")
print(f"Example of the first chunk:\n{chunks[0].page_content[:200]}...")
print(f"Metadata of the chunk: {chunks[0].metadata}")

Total chunks created: 377
Example of the first chunk:
Produced by Charles Keller (for Tina); and David Widger

Dashes; small checks; quick pass; gutchecked twice; jeebies; spellcheck

THE ADVENTURES OF PINOCCHIO

by C. Collodi

[Pseudonym of Carlo Lorenz...
Metadata of the chunk: {'source': 'adventures_of_pinocchio_collodi.txt', 'title': 'The Adventures of Pinocchio', 'author': 'Carlo Collodi', 'start_index': 0}


**4. Embed and store.**

In [5]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    collection_name="gutenberg_rag",
    embedding_function=embedding_model,
    persist_directory="chroma_db"
)

# Check if the storage is empty, to avoid re-embedding
if vectorstore._collection.count() == 0:
    chunk_ids = [f"{BOOK}-{i}" for i in range(len(chunks))]
    vectorstore.add_documents(documents=chunks, ids=chunk_ids)
    print(f"Added {len(chunks)} documents to Chroma.")
else:
    print(f"Collection already contains {vectorstore._collection.count()} records. Skipping addition.")

/Users/khaldouna/Documents/Neuefische/w6d4/exercise/presentation/project-book-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 12:54:03.481000 63457 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12228.64it/s]


Collection already contains 377 records. Skipping addition.


**5. Retrieve.**

In [6]:
# create a retriever for the 4 nearest fragments
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Check several questions
questions = [
    "Why is the carpenter called Mastro Cherry?",
    "What happens to Pinocchio's nose when he lies?",
    "What did Geppetto sell to buy Pinocchio an A-B-C book?"
]

for q in questions:
    print("=" * 70)
    print(f"QUESTION: {q}")
    print("=" * 70)
    
    docs = retriever.invoke(q)
    
    for i, chunk in enumerate(docs, start=1):
        start_idx = chunk.metadata.get("start_index", "Not specified")
        print(f"\n[Fragment {i} | start_index: {start_idx}]")
        # Print the first 250 characters of the fragment
        print(chunk.page_content[:250].replace("\n", " ") + "...")
    print("\n")

QUESTION: Why is the carpenter called Mastro Cherry?

[Fragment 1 | start_index: 1024]
As soon as he saw that piece of wood, Mastro Cherry was filled with joy. Rubbing his hands together happily, he mumbled half to himself:  “This has come in the nick of time. I shall use it to make the leg of a table.”  He grasped the hatchet quickly ...

[Fragment 2 | start_index: 374]
Centuries ago there lived--  “A king!” my little readers will say immediately.  No, children, you are mistaken. Once upon a time there was a piece of wood. It was not an expensive piece of wood. Far from it. Just a common block of firewood, one of th...

[Fragment 3 | start_index: 2122]
“Oh, oh! You hurt!” cried the same far-away little voice.  Mastro Cherry grew dumb, his eyes popped out of his head, his mouth opened wide, and his tongue hung down on his chin.  As soon as he regained the use of his senses, he said, trembling and st...

[Fragment 4 | start_index: 3766]
CHAPTER 2  Mastro Cherry gives the piece of wood t

**6. Generate**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Prompt template
prompt_template = ChatPromptTemplate.from_template("""
You are a library reading companion. Answer the user's question relying ONLY on the provided book passages below.
If the answer cannot be found in the passages, you MUST state clearly: "I cannot find the answer in the provided passages of the book."
Do not make up facts or use outside knowledge.

Passages from the book:
{context}

Question:
{question}

Answer:
""")

# init ChatGroq
# if model "openai/gpt-oss-20b" will give a missing error, replace with "llama-3.1-8b-instant"
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# get LCEL chain
parser = StrOutputParser()
chain = prompt_template | llm | parser

# ask(question)
def ask(question: str):
    # get 4 relevant documents
    docs = retriever.invoke(question)
    
    # Combine them into a single context string
    context_text = "\n\n---\n\n".join([doc.page_content for doc in docs])
    
    # Generate answer
    answer = chain.invoke({
        "context": context_text,
        "question": question
    })
    
    # Extract start_index of each fragment
    indices = [doc.metadata.get("start_index") for doc in docs]
    
    return answer, indices, docs

# Initial Testing questions
test_questions = [
    "Why is the carpenter called Mastro Cherry?",
    "What did Geppetto sell to buy Pinocchio an A-B-C book?",
    "What kind of car does Geppetto drive?"  # question that isn’t in the book!!!
]

for q in test_questions:
    ans, start_indices, docs = ask(q)
    print("=" * 70)
    print(f"QUESTION: {q}")
    print(f"ANSWER:\n{ans}")
    print(f"PASSAGE INDICES (start_index): {start_indices}")
    print("=" * 70 + "\n")

QUESTION: Why is the carpenter called Mastro Cherry?
ANSWER:
The carpenter is called Mastro Cherry because his nose is round, red, and shiny, resembling a ripe cherry.
PASSAGE INDICES (start_index): [1024, 374, 2122, 3766]

QUESTION: What did Geppetto sell to buy Pinocchio an A-B-C book?
ANSWER:
Geppetto sold his coat to buy Pinocchio an A‑B‑C book.
PASSAGE INDICES (start_index): [28278, 32915, 29518, 41087]

QUESTION: What kind of car does Geppetto drive?
ANSWER:
I cannot find the answer in the provided passages of the book.
PASSAGE INDICES (start_index): [7574, 3766, 7113, 28278]



In [ ]:
chain

langchain_core.runnables.base.RunnableSequence

**7. Scroing**

In [16]:
def format_passages(chunks):
    """Number the retrieved chunks, as the prompt expects."""
    return "\n\n".join(
        f"[{i}] {chunk.page_content}" for i, chunk in enumerate(chunks, start=1)
    )

def score(verdicts, good_label):
    """Fraction of the judge's lines carrying the good label, and the lines themselves."""
    lines = [line.strip() for line in verdicts.strip().split("\n") if "|" in line]
    return sum(line.upper().startswith(good_label) for line in lines) / len(lines), lines

question = "Why is the carpenter called Mastro Cherry?"
# what a correct answer would say, written from the handbook by hand
reference = "Because the tip of his nose was so red and shiny that it looked like a ripe cherry."

passages = format_passages(retriever.invoke(question))
answer = chain.invoke({'context':passages, 'question': question})

**Faithfulness**

In [17]:
# one line per claim, so the verdicts can be counted without reading prose
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You check an answer against the passages it was given. Split the answer into "
            "single factual claims and judge each one on the passages alone. Write one line "
            "per claim: SUPPORTED or UNSUPPORTED, then ' | ', then the claim. Nothing else.",
        ),
        ("human", "Passages:\n{passages}\n\nAnswer:\n{answer}"),
    ]
)

faithfulness, claims = score(
    (judge_prompt | llm | StrOutputParser()).invoke(
        {"passages": passages, "answer": answer}
    ),
    "SUPPORTED",
)
print(f"faithfulness {faithfulness:.2f}", *claims, sep="\n")

faithfulness 1.00
SUPPORTED | The carpenter is called Mastro Cherry because his nose is round, red, and shiny, resembling a ripe cherry.


**Recall**

In [18]:
# the same line format, but judged against the reference instead of the answer
recall_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You check whether passages contain what a correct answer needs. Split the "
            "correct answer into single factual claims and look for each one in the "
            "passages. Write one line per claim: FOUND or MISSING, then ' | ', then the "
            "claim. Nothing else.",
        ),
        ("human", "Passages:\n{passages}\n\nCorrect answer:\n{reference}"),
    ]
)

recall, needed = score(
    (recall_prompt | llm | StrOutputParser()).invoke(
        {"passages": passages, "reference": reference}
    ),
    "FOUND",
)
print(f"context recall {recall:.2f}", *needed, sep="\n")

context recall 1.00
FOUND | tip of his nose was red
FOUND | tip of his nose was shiny
FOUND | tip of his nose looked like a ripe cherry


In [19]:
def evaluate(question, reference, verbose=True):
    """Run the whole RAG pipeline for one question and score it.

    Wraps the pieces defined above: retriever -> format_passages -> chain for the answer,
    then judge_prompt (faithfulness) and recall_prompt (context recall), counted with score().
    """
    passages = format_passages(retriever.invoke(question))
    answer = chain.invoke({"context": passages, "question": question})

    judge = StrOutputParser()
    faith_raw = (judge_prompt | llm | judge).invoke({"passages": passages, "answer": answer})
    recall_raw = (recall_prompt | llm | judge).invoke({"passages": passages, "reference": reference})

    # score() divides by the number of lines, so guard against a judge that returns none
    try:
        faithfulness, claims = score(faith_raw, "SUPPORTED")
    except ZeroDivisionError:
        faithfulness, claims = float("nan"), []
    try:
        recall, needed = score(recall_raw, "FOUND")
    except ZeroDivisionError:
        recall, needed = float("nan"), []

    if verbose:
        print("=" * 70)
        print(f"QUESTION: {question}")
        print(f"ANSWER:   {answer.strip()}")
        print(f"\nfaithfulness {faithfulness:.2f}", *claims, sep="\n  ")
        print(f"\ncontext recall {recall:.2f}", *needed, sep="\n  ")

    return {
        "question": question,
        "answer": answer,
        "faithfulness": faithfulness,
        "recall": recall,
        "claims": claims,
        "needed": needed,
    }


# Five tests. The references are written by hand from the book and are deliberately
# demanding for recall: each needs several facts, and most of those facts sit in
# different chapters, so 4 retrieved chunks of 700 characters may not cover them all.
tests = [
    # two hops: Geppetto's sacrifice (ch. 9) and what Pinocchio did with the book
    (
        "What did Geppetto sell to buy Pinocchio an A-B-C book, and what did Pinocchio do with the book?",
        "Geppetto sold his old coat to buy the A-B-C book and came home in his shirt sleeves in the cold. "
        "Pinocchio sold the book for four pennies to buy a ticket to the Marionette Theater.",
    ),
    # a chain of events spread over several chapters
    (
        "How many gold pieces did Fire Eater give Pinocchio, and what happened to them?",
        "Fire Eater gave Pinocchio five gold pieces for his father Geppetto. "
        "The Fox and the Cat persuaded him to bury them in the Field of Wonders so they would grow into a tree of gold. "
        "While Pinocchio was away, the Fox and the Cat dug up the gold pieces and stole them.",
    ),
    # paraphrased question: none of the book's key words ("woodpeckers", "Fairy") appear in it
    (
        "After his face changed because he told untruths, how was it put back to normal?",
        "Pinocchio's nose grew longer with every lie he told the Fairy. "
        "The Fairy clapped her hands and a thousand woodpeckers flew in and pecked his nose back to its normal size.",
    ),
    # a long arc near the end of the book
    (
        "What happened to Pinocchio and Lamp-Wick in the Land of Toys?",
        "Pinocchio ran away with his friend Lamp-Wick to the Land of Toys, where there was no school and every day was play. "
        "After some months they both grew donkey ears and turned into real donkeys. "
        "Pinocchio was sold to the owner of a circus.",
    ),
    # the reunion: needs the Shark, how long Geppetto was inside, and the escape
    (
        "How does Pinocchio find Geppetto again, and how do they escape?",
        "Pinocchio is swallowed by the Terrible Shark and finds Geppetto living inside it. "
        "Geppetto had been inside the Shark for two years. "
        "Pinocchio carries Geppetto on his back out of the Shark's mouth while it sleeps, "
        "and the Tunny carries them to the shore.",
    ),
]

results = [evaluate(q, ref) for q, ref in tests]

# summary table
print("\n" + "=" * 70)
print(f"{'#':<3}{'faithfulness':>13}{'recall':>9}   question")
for i, r in enumerate(results, start=1):
    print(f"{i:<3}{r['faithfulness']:>13.2f}{r['recall']:>9.2f}   {r['question'][:50]}")
n = len(results)
print(f"{'avg':<3}{sum(r['faithfulness'] for r in results) / n:>13.2f}"
      f"{sum(r['recall'] for r in results) / n:>9.2f}")


QUESTION: What did Geppetto sell to buy Pinocchio an A-B-C book, and what did Pinocchio do with the book?
ANSWER:   Geppetto sold his coat to buy Pinocchio an A‑B‑C book.  
Pinocchio later sold that very book in order to pay for his way into the Marionette Theater.

faithfulness 1.00
  SUPPORTED | Geppetto sold his coat to buy Pinocchio an A‑B‑C book.
  SUPPORTED | Pinocchio later sold that very book in order to pay for his way into the Marionette Theater.

context recall 1.00
  FOUND | Geppetto sold his old coat to buy the A-B-C book
  FOUND | Geppetto came home in his shirt sleeves in the cold
  FOUND | Pinocchio sold the book for four pennies to buy a ticket to the Marionette Theater
QUESTION: How many gold pieces did Fire Eater give Pinocchio, and what happened to them?
ANSWER:   Fire Eater gave Pinocchio **five gold pieces**.  
Later, Pinocchio told the Fairy that he had lost them, but in truth he **swallowed the gold pieces when he drank the medicine**.

faithfulness 1.00
  SUPPO